In [69]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
import joblib

def ft_train():
    data = pd.read_csv("data.csv")
    data = data[data["prix_eur"]<5000000]
    
    X = data.drop(columns=["Unnamed: 0", "prix_eur" , "prix_m2" , "url" , "latitude" , "longitude" , "adresse" ])
    y = data["prix_eur"]
    
    
    mapping_dpe = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
    X["dpe"] = X["dpe"].map(mapping_dpe)
    
    le_etage = LabelEncoder()
    X["etage"] = le_etage.fit_transform(X["etage"])
    joblib.dump(le_etage , "utils/le_etage.joblib")


    
    le_type_bien = LabelEncoder()
    X["type_bien"] = le_type_bien.fit_transform(X["type_bien"])
    joblib.dump(le_type_bien , "utils/le_type_bien.joblib")
    
    

    X = X.fillna(-10000)
    
    
    model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.01,
                                       max_depth=5, random_state=42)
    
    model.fit(X,y)

        
    joblib.dump(model , "utils/model.joblib")


ft_train()
    

In [50]:
data = pd.read_csv("data.csv")
    
    
X = data.drop(columns=["Unnamed: 0", "prix_eur" , "prix_m2" , "url" , "latitude" , "longitude" , "adresse" ])
y = data["prix_eur"]

In [53]:
def ft_pred(type_bien, surface_m2, 
            nb_pieces, nb_chambres, etage,
           dpe , code_postal):

    

    le_etage = joblib.load("utils/le_etage.joblib")
    etage = le_etage.transform([etage])[0]

    le_type_bien = joblib.load("utils/le_type_bien.joblib")
    type_bien = le_type_bien.transform([type_bien])[0]

    mapping_dpe = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
    dpe = mapping_dpe[dpe]

    model = joblib.load("utils/model.joblib")

    estimation = model.predict([[type_bien, surface_m2, 
            nb_pieces, nb_chambres, etage,
           dpe , code_postal]])

    return(estimation[0])
    

In [70]:
print(ft_pred(
'Appartement à vendre' , 
50 , 
3 , 
2 , 
'1 étages', 
"A" , 
75011.0))

578510.3466043838


/Users/cours/.conda/envs/env_mlp/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [72]:
model = joblib.load("utils/model.joblib")
model.fe

In [74]:
model.feature_names_in_

array(['type_bien', 'surface_m2', 'nb_pieces', 'nb_chambres', 'etage',
       'dpe', 'code_postal'], dtype=object)